# Section 1: Fundamentals of Generative AI
## GCP Generative AI Leader Certification

This notebook covers hands-on exercises for generative AI fundamentals on Google Cloud:
- Vertex AI SDK setup and authentication
- Exploring Google foundation models (Gemini, Gemma, Imagen)
- Text generation basics with Gemini
- Multimodal input (text + image)
- Comparing model variants (Pro vs Flash)
- Understanding embeddings and vector representations
- ML lifecycle demonstration

**Prerequisites**: A GCP project with Vertex AI API enabled and billing configured.

---
## 1. Environment Setup

In [ ]:
# Install the Vertex AI SDK
!pip install -q google-cloud-aiplatform>=1.60.0 google-generativeai>=0.7.0

In [ ]:
# Authenticate (Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
except ImportError:
    print("Not in Colab. Using default credentials.")

# Set your project and location
PROJECT_ID = "your-project-id"  # <-- Replace with your GCP project ID
LOCATION = "us-central1"

import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"Vertex AI initialized: project={PROJECT_ID}, location={LOCATION}")

In [ ]:
# Import generative model classes
from vertexai.generative_models import GenerativeModel, GenerationConfig, Part
print("Imports successful.")

---
## 2. Exploring Google Foundation Models

Google offers multiple foundation model families:
- **Gemini** (Pro, Flash, Ultra, Nano) - multimodal, most capable
- **Gemma** (2B, 7B) - open-weight, self-deployable
- **Imagen** - text-to-image diffusion model
- **Veo** - video generation
- **Chirp** - speech foundation model

Let's explore the available models through Vertex AI.

In [ ]:
# List available Gemini model variants
try:
    from google.cloud import aiplatform
    aiplatform.init(project=PROJECT_ID, location=LOCATION)

    # List publisher models matching 'gemini'
    models = aiplatform.Model.list(filter='display_name~gemini')
    print(f"Found {len(models)} Gemini-related models")
    for m in models[:5]:
        print(f"  - {m.display_name}")
except Exception as e:
    print(f"Note: {e}")
    print("\n--- Mock Output ---")
    print("Available Gemini models on Vertex AI:")
    print("  - gemini-1.5-pro-002     (balanced quality & speed)")
    print("  - gemini-1.5-flash-002   (fastest, lowest cost)")
    print("  - gemini-1.5-pro         (latest stable)")
    print("  - gemini-1.5-flash       (latest stable flash)")
    print("  - gemini-1.0-pro         (previous generation)")

### Model Selection Guide

| Model | Best For | Context Window | Relative Cost |
|-------|----------|---------------|---------------|
| **Gemini Ultra** | Complex reasoning, research | 1M tokens | Highest |
| **Gemini Pro** | General-purpose, balanced | 2M tokens | Medium |
| **Gemini Flash** | High-volume, low-latency | 1M tokens | Lowest |
| **Gemini Nano** | On-device (Pixel phones) | Limited | Free (on-device) |
| **Gemma 7B** | Self-hosted, fine-tuning | 8K tokens | Self-managed |

---
## 3. Text Generation with Gemini

Let's explore basic text generation and understand the difference
between generative and discriminative AI.

In [ ]:
# Initialize Gemini Pro model
try:
    model = GenerativeModel("gemini-1.5-pro")

    # GENERATIVE task: Create new content
    gen_response = model.generate_content(
        "Explain the difference between generative AI and discriminative AI "
        "in 4 concise bullet points suitable for a business leader."
    )
    print("=== Generative Task: Explain a Concept ===")
    print(gen_response.text)

except Exception as e:
    print(f"API call note: {e}")
    print("\n--- Mock Output ---")
    print("=== Generative Task: Explain a Concept ===")
    print("- **Discriminative AI** classifies or labels data (e.g., spam detection, image recognition)")
    print("- **Generative AI** creates new content (text, images, code, video)")
    print("- Discriminative models answer 'What is this?'; Generative models answer 'Create something like this'")
    print("- Modern gen AI (Gemini) can do both: generate content AND classify/analyze it")

In [ ]:
# DISCRIMINATIVE-style task using a generative model
# Gemini can classify, even though it's generative
try:
    classification_prompt = """Classify each item as STRUCTURED or UNSTRUCTURED data:

1. A MySQL database of customer orders
2. A collection of customer support call recordings
3. A CSV file with stock prices
4. A folder of product photos
5. A JSON API response with user profiles

Return a numbered list with the classification."""

    response = model.generate_content(classification_prompt)
    print("=== Discriminative-style Task: Classification ===")
    print(response.text)

except Exception as e:
    print(f"API call note: {e}")
    print("\n--- Mock Output ---")
    print("=== Discriminative-style Task: Classification ===")
    print("1. MySQL database of customer orders - STRUCTURED")
    print("2. Customer support call recordings - UNSTRUCTURED")
    print("3. CSV file with stock prices - STRUCTURED")
    print("4. Folder of product photos - UNSTRUCTURED")
    print("5. JSON API response with user profiles - STRUCTURED (semi-structured)")

---
## 4. Multimodal Input (Text + Image)

Gemini is **natively multimodal** - it understands text, images, audio,
and video in a single model. Let's demonstrate image understanding.

In [ ]:
import requests
from IPython.display import Image, display

# Download a sample image
image_url = "https://storage.googleapis.com/cloud-samples-data/ai-platform/flowers/daisy/100080576_f52e8ee070_n.jpg"
try:
    image_bytes = requests.get(image_url).content
    display(Image(data=image_bytes, width=300))
    print("Image loaded successfully.")
except Exception as e:
    print(f"Could not load image: {e}")
    image_bytes = None

In [ ]:
# Multimodal prompt: text + image
try:
    if image_bytes:
        image_part = Part.from_data(data=image_bytes, mime_type="image/jpeg")
        multimodal_response = model.generate_content([
            image_part,
            "Describe this image. What type of flower is it? What data type would this "
            "be classified as in a machine learning context (structured, semi-structured, or unstructured)?"
        ])
        print("=== Multimodal Response ===")
        print(multimodal_response.text)
    else:
        print("No image available for multimodal test.")

except Exception as e:
    print(f"API call note: {e}")
    print("\n--- Mock Output ---")
    print("=== Multimodal Response ===")
    print("This image shows a daisy flower with white petals and a yellow center.")
    print("It appears to be in a garden setting with green foliage in the background.")
    print("\nIn ML terms, this image is UNSTRUCTURED data - it has no predefined")
    print("schema or tabular format. Processing it requires deep learning models")
    print("(CNNs or vision transformers) rather than traditional ML approaches.")

---
## 5. Comparing Model Variants: Pro vs Flash

A key exam topic is knowing when to choose which model variant.
Let's compare Gemini Pro and Flash on the same task.

In [ ]:
import time

comparison_prompt = (
    "Explain the ML lifecycle in 5 steps. For each step, give a one-sentence "
    "description and name one Google Cloud tool that supports it."
)

results = {}
for model_name in ["gemini-1.5-pro", "gemini-1.5-flash"]:
    try:
        m = GenerativeModel(model_name)
        start = time.time()
        resp = m.generate_content(comparison_prompt)
        elapsed = time.time() - start
        results[model_name] = {
            "text": resp.text[:300] + "...",
            "time": f"{elapsed:.2f}s",
            "tokens": getattr(resp.usage_metadata, 'total_token_count', 'N/A')
        }
    except Exception as e:
        results[model_name] = {"text": f"(API unavailable: {e})", "time": "N/A", "tokens": "N/A"}

for name, data in results.items():
    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"Latency: {data['time']} | Tokens: {data['tokens']}")
    print(f"Response: {data['text']}")

if all(v['time'] == 'N/A' for v in results.values()):
    print("\n--- Mock Comparison ---")
    print("gemini-1.5-pro:   Latency ~2.1s  | Higher quality, more detail")
    print("gemini-1.5-flash: Latency ~0.8s  | Good quality, much faster")
    print("\nExam tip: Use Flash for high-volume, latency-sensitive tasks.")
    print("Use Pro for complex reasoning requiring maximum quality.")

---
## 6. Text Embeddings

Embeddings are vector representations of text that capture semantic meaning.
They are the foundation of RAG systems (covered in Section 3).
Similar texts have similar embedding vectors.

In [ ]:
try:
    from vertexai.language_models import TextEmbeddingModel
    import numpy as np

    embed_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

    # Embed some sample texts
    texts = [
        "Generative AI creates new content like text and images.",
        "LLMs generate text by predicting the next token.",
        "The weather in Tokyo is sunny today.",
        "Foundation models are pre-trained on massive datasets."
    ]

    embeddings = embed_model.get_embeddings(texts)
    vectors = [np.array(e.values) for e in embeddings]

    print(f"Embedding dimension: {len(vectors[0])}")
    print(f"\nSemantic similarity (cosine):")

    for i in range(len(texts)):
        for j in range(i+1, len(texts)):
            cos_sim = np.dot(vectors[i], vectors[j]) / (np.linalg.norm(vectors[i]) * np.linalg.norm(vectors[j]))
            print(f"  [{i}] vs [{j}]: {cos_sim:.4f}  ({texts[i][:40]}... vs {texts[j][:40]}...)")

except Exception as e:
    print(f"Embedding API note: {e}")
    print("\n--- Mock Output ---")
    print("Embedding dimension: 768")
    print("\nSemantic similarity (cosine):")
    print("  [0] vs [1]: 0.8234  (GenAI vs LLMs - HIGH similarity, related topics)")
    print("  [0] vs [2]: 0.2145  (GenAI vs Weather - LOW similarity, unrelated)")
    print("  [0] vs [3]: 0.7891  (GenAI vs Foundation models - HIGH, related)")
    print("  [1] vs [2]: 0.1987  (LLMs vs Weather - LOW, unrelated)")
    print("  [1] vs [3]: 0.8102  (LLMs vs Foundation models - HIGH, related)")
    print("  [2] vs [3]: 0.1654  (Weather vs Foundation models - LOW, unrelated)")
    print("\nKey insight: Embeddings capture SEMANTIC meaning, not just keywords.")
    print("This is what powers RAG retrieval - finding relevant documents by meaning.")

---
## 7. Understanding Tokens

Tokens are the basic units that LLMs process. Understanding tokenization
is essential for prompt design, cost estimation, and context window management.

In [ ]:
# Count tokens for different inputs
try:
    model = GenerativeModel("gemini-1.5-pro")

    test_texts = [
        "Hello, world!",
        "Generative AI is a subset of deep learning that creates new content.",
        "The Google Cloud Professional Machine Learning Engineer certification " * 10,
    ]

    print("=== Token Counting ===")
    for text in test_texts:
        resp = model.count_tokens(text)
        chars = len(text)
        ratio = chars / resp.total_tokens if resp.total_tokens > 0 else 0
        print(f"  Chars: {chars:>5} | Tokens: {resp.total_tokens:>4} | Chars/Token: {ratio:.1f}")
        print(f"  Text: {text[:60]}...\n")

except Exception as e:
    print(f"Token counting note: {e}")
    print("\n--- Mock Output ---")
    print("=== Token Counting ===")
    print("  Chars:    13 | Tokens:    4 | Chars/Token: 3.3")
    print("  Text: Hello, world!")
    print("")
    print("  Chars:    67 | Tokens:   14 | Chars/Token: 4.8")
    print("  Text: Generative AI is a subset of deep learning that creates...")
    print("")
    print("  Chars:   680 | Tokens:  130 | Chars/Token: 5.2")
    print("  Text: The Google Cloud Professional Machine Learning Engineer...")
    print("\nRule of thumb: 1 token ~ 4 characters or ~3/4 of a word in English.")
    print("Gemini 1.5 Pro context window: 2,000,000 tokens (~1.5M words).")

---
## 8. Data Quality Impact on Generation

Let's demonstrate how data quality affects generative AI outputs,
reinforcing the data fundamentals from the study guide.

In [ ]:
# Demonstrate how context quality affects output quality
try:
    model = GenerativeModel("gemini-1.5-pro")

    # HIGH quality context
    good_context = """Company Policy (Updated January 2025):
    - Return period: 30 days from purchase date
    - Full refund for unopened items
    - 80% refund for opened items in original condition
    - No refund after 30 days
    - Contact: returns@example.com"""

    # LOW quality context (outdated, inconsistent)
    bad_context = """Company Policy (maybe from 2019?):
    - Returns might be 30 days or 60 days depending
    - Refund amount varies
    - Actually we might not do refunds anymore
    - Contact: unknown"""

    question = "What is the return policy?"

    for label, ctx in [("HIGH-quality data", good_context), ("LOW-quality data", bad_context)]:
        prompt = f"Based on this context, answer the question.\n\nContext: {ctx}\n\nQuestion: {question}"
        response = model.generate_content(prompt)
        print(f"\n=== {label} ===")
        print(response.text[:300])

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("\n=== HIGH-quality data ===")
    print("Our return policy allows returns within 30 days of purchase.")
    print("Unopened items receive a full refund. Opened items in original")
    print("condition receive an 80% refund. Contact returns@example.com.")
    print("\n=== LOW-quality data ===")
    print("The return policy is unclear. It may be 30 or 60 days, and")
    print("refund amounts vary. There's some uncertainty about whether")
    print("refunds are currently offered. I'd recommend contacting support.")
    print("\n** Key takeaway: Data quality directly impacts output quality. **")
    print("This is why the exam emphasizes data accuracy, completeness,")
    print("timeliness, and consistency as prerequisites for good gen AI.")

---
## Summary

In this notebook, you explored:

1. **SDK Setup** - Initializing Vertex AI and importing model classes
2. **Google Models** - Exploring the Gemini family (Pro, Flash, Ultra, Nano) and other models (Gemma, Imagen, Veo)
3. **Text Generation** - Using Gemini for generative and discriminative-style tasks
4. **Multimodal** - Sending images + text to Gemini for visual understanding
5. **Model Comparison** - Pro vs Flash: quality vs speed tradeoffs
6. **Embeddings** - Vector representations that power semantic search and RAG
7. **Tokens** - Understanding tokenization for cost and context management
8. **Data Quality** - How data quality directly impacts generative AI output quality

**Key exam takeaways:**
- AI > ML > Deep Learning > Generative AI (nested relationship)
- Gemini = proprietary, managed, most capable; Gemma = open-weight, self-deployable
- Generative models can also perform discriminative tasks
- Data quality (accuracy, completeness, timeliness) is critical for gen AI

**Next**: [Section 2 - Google Cloud's Gen AI Offerings](02-gcp-gen-ai-offerings.ipynb)